In [ ]:
import torch
import torch.nn.functional as F

from src.temporal_score_cam import TemporalScoreCAM
from src.utils.config_utils import setup_config_dictionary
from src.utils.save_utils import load_model
from src.utils.plot_utils import plot_frames, plot_cams, plot_combined_frames, combine_polarities
from src.utils.snn_utils import compute_raw_mem, compute_delta_raw_mem

from src.utils.data_utils import get_dataloaders
from src.utils.model_utils import get_model

config = setup_config_dictionary()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(torch.seed())
# torch.manual_seed(4762538657146009718)

In [ ]:
gesture_map = {
    0: "hand_clapping",
    1: "right_hand_wave",
    2: "left_hand_wave",
    3: "right_hand_clockwise",
    4: "right_hand_counter_clockwise",
    5: "left_hand_clockwise",
    6: "left_hand_counter_clockwise",
    7: "forearm_roll_forward",
    8: "forearm_roll_backward",
    9: "drums",
    10: "guitar",
    #11: "random_other_gestures", # -> not implemented here
}

In [ ]:
train_loader, test_loader = get_dataloaders(
    config["DATASET_NAME"], 
    config["BATCH_SIZE"], 
    False,
)

event_batches, _ = next(iter(train_loader))
num_channels = event_batches.shape[2]

# load the model
model = get_model(
        config["MODEL_NAME"],
        num_classes=len(train_loader.dataset.classes),
        time_steps=config["NUM_STEPS"],
        num_channels=num_channels
    ).to(device)

load_model(model, config["MODEL_SAVE_DIR"], model_name="vgg11", dataset_name="dvs_gesture")


# Visualize using our ScoreCAM implementation
score_cam = TemporalScoreCAM(model)
conv_layers = [module for module in model.modules() if isinstance(module, torch.nn.Conv2d)]
print("Total convolutional layers:", len(conv_layers))

model.eval()
with torch.no_grad():
    for i, (event_batches, labels) in enumerate(test_loader):
        event_batches = event_batches.to(device)
        labels = labels.to(device)
        if isinstance(model, tuple):
                # output would be out_spikes
                out_spikes, out_mem = model(event_batches)
                raw_mem = compute_raw_mem(out_spikes, out_mem)
                delta_raw_mem = compute_delta_raw_mem(raw_mem)
                output_class = F.softmax(out_spikes.sum(dim=1), dim=1).cpu().numpy()
        else:
            # output would be out_classes or out_membrane. It will be matched with appropriate criterion
            outputs = model(event_batches)
            delta_outputs = compute_delta_raw_mem(outputs)
            # only take the last time step for classification for the membrane potential-based models
            output_class = F.softmax(outputs[:, -1, :], dim=1).cpu().numpy()

        predicted_label = int(output_class.argmax(axis=1)[0])
        print(f"Output probabilities: {[f'{p:.2%}' for p in output_class[0]]}\nFrom spike sum {out_spikes.sum(dim=1)}")
        print(f"Predicted label {predicted_label},real label: {labels.item()}\nPredicted gesture: {gesture_map[predicted_label]}, real gesture: {gesture_map[labels.item()]}")
        combined_frames = combine_polarities(event_batches[0].cpu().numpy())
        # Compute ScoreCAM
        #plot_frames(event_batches[0].cpu().numpy())
        plot_combined_frames(combined_frames)
        print("Real label vs predicted label cams for different CONV layers:")
        
        target_layer = conv_layers[-5]
        cam = score_cam.generate_cam(event_batches, target_layer=target_layer)
        print("Layer 4:")
        plot_cams(cam, class_idx=predicted_label, frames=combined_frames)
        plot_cams(cam, class_idx=labels.item(), frames=combined_frames)
        
        target_layer = conv_layers[-3]
        cam = score_cam.generate_cam(event_batches, target_layer=target_layer)
        print("Layer 6:")
        plot_cams(cam, class_idx=predicted_label, frames=combined_frames)
        plot_cams(cam, class_idx=labels.item(), frames=combined_frames)

        target_layer = conv_layers[-1]
        cam = score_cam.generate_cam(event_batches, target_layer=target_layer)
        print("Layer 8:")
        plot_cams(cam, class_idx=predicted_label, frames=combined_frames)
        plot_cams(cam, class_idx=labels.item(), frames=combined_frames)

        #print(f"Cams for random label, e.g. {0}, correspoding to gesture {gesture_map[0]}:")
        #plot_cams(cam, class_idx=0)
        # only one sample for now
        break
